# Clarté Commerce — RFM Segmentation

**Client:** Clarté Commerce S.A.S.  
**Analyst:** Nurbol Sultanov  
**Date:** February 2026  

RFM (Recency, Frequency, Monetary) segmentation of the customer base.  
Goal: compare segment distribution before and after the Q3 2023 rebranding.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

sys.path.append('../src')
from rfm_utils import calculate_rfm, assign_rfm_labels

REBRAND_DATE = pd.Timestamp('2023-08-15')

In [2]:
transactions = pd.read_csv('../data/raw/transactions.csv', parse_dates=['transaction_date'])
customers = pd.read_csv('../data/raw/customers.csv', parse_dates=['registration_date'])

print(f'Transactions: {transactions.shape}')
print(f'Customers: {customers.shape}')

Transactions: (92330, 11)
Customers: (12005, 9)


In [3]:
mask = transactions['customer_id'].str.contains('TEST')
print(f'Filtered out {mask.sum()} test transactions')
transactions = transactions[~mask]
print(f'Working with {len(transactions):,} transactions')

Filtered out 20 test transactions
Working with 92,310 transactions


## 1. Full-Period RFM

In [4]:
snapshot_date = pd.Timestamp('2024-12-31')

rfm = calculate_rfm(transactions, snapshot_date)

print(f'RFM table: {len(rfm):,} customers')
print()
print(rfm[['recency', 'frequency', 'monetary']].describe().loc[['mean','std','min','50%','max']].round(2).to_string())

RFM table: 11,572 customers

         recency  frequency    monetary
mean      243.52       7.98      886.43
std       195.34       5.62      724.89
min         0.00       1.00        4.53
50%       189.00       6.00      672.80
max       730.00      42.00     6842.10


In [5]:
# Assign quintile scores
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm['r_score'] = rfm['r_score'].astype(int)
rfm['f_score'] = rfm['f_score'].astype(int)
rfm['m_score'] = rfm['m_score'].astype(int)
rfm['rfm_total'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

rfm = assign_rfm_labels(rfm)

In [6]:
# Segment distribution
segment_order = ['Champions', 'Loyal Customers', 'Potential Loyalists', 
                 'New Customers', 'At Risk', 'Cant Lose Them', 'Lost', 'Other']

colors_map = {
    'Champions': '#27ae60',
    'Loyal Customers': '#2ecc71',
    'Potential Loyalists': '#3498db',
    'New Customers': '#1abc9c',
    'At Risk': '#e67e22',
    'Cant Lose Them': '#e74c3c',
    'Lost': '#95a5a6',
    'Other': '#bdc3c7'
}

seg_counts = rfm['rfm_label'].value_counts().reindex(segment_order).dropna()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(seg_counts.index, seg_counts.values,
               color=[colors_map[s] for s in seg_counts.index], edgecolor='white')
ax.set_title('Customer Segment Distribution (Full Period)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Customers')

for bar, val in zip(bars, seg_counts.values):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({val/len(rfm):.1%})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/rfm_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Pre vs Post Rebrand RFM Comparison

Split transactions at the rebrand date and compute separate RFM for each period.

In [7]:
pre_txn = transactions[transactions['transaction_date'] < REBRAND_DATE]
post_txn = transactions[transactions['transaction_date'] >= REBRAND_DATE]

print(f'Pre-rebrand:  {len(pre_txn):,} transactions, {pre_txn["customer_id"].nunique():,} customers')
print(f'Post-rebrand: {len(post_txn):,} transactions, {post_txn["customer_id"].nunique():,} customers')

Pre-rebrand:  51,938 transactions, 9,842 customers
Post-rebrand: 40,372 transactions, 8,214 customers


In [8]:
# Pre-rebrand RFM (snapshot = rebrand date)
rfm_pre = calculate_rfm(pre_txn, REBRAND_DATE)
rfm_pre['r_score'] = pd.qcut(rfm_pre['recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm_pre['f_score'] = pd.qcut(rfm_pre['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_pre['m_score'] = pd.qcut(rfm_pre['monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_pre['rfm_total'] = rfm_pre['r_score'] + rfm_pre['f_score'] + rfm_pre['m_score']
rfm_pre = assign_rfm_labels(rfm_pre)

# Post-rebrand RFM (snapshot = end of data)
rfm_post = calculate_rfm(post_txn, snapshot_date)
rfm_post['r_score'] = pd.qcut(rfm_post['recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm_post['f_score'] = pd.qcut(rfm_post['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_post['m_score'] = pd.qcut(rfm_post['monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_post['rfm_total'] = rfm_post['r_score'] + rfm_post['f_score'] + rfm_post['m_score']
rfm_post = assign_rfm_labels(rfm_post)

In [9]:
# Compare segment distributions
pre_seg = rfm_pre['rfm_label'].value_counts(normalize=True).reindex(segment_order).fillna(0)
post_seg = rfm_post['rfm_label'].value_counts(normalize=True).reindex(segment_order).fillna(0)

comparison = pd.DataFrame({
    'Pre-Rebrand': pre_seg,
    'Post-Rebrand': post_seg
})
comparison['Change (pp)'] = ((post_seg - pre_seg) * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(segment_order))
width = 0.35

bars1 = ax.bar(x - width/2, pre_seg.values * 100, width, label='Pre-Rebrand', color='#3498db')
bars2 = ax.bar(x + width/2, post_seg.values * 100, width, label='Post-Rebrand', color='#e74c3c')

ax.set_ylabel('Share of Customers (%)')
ax.set_title('RFM Segment Distribution \u2014 Pre vs Post Rebrand', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(segment_order, rotation=25, ha='right')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/rfm_pre_post_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [10]:
print('Segment shift (percentage points):\n')
for seg in segment_order:
    if seg == 'Other':
        continue
    change = comparison.loc[seg, 'Change (pp)']
    print(f'{seg:22s} {change:+.1f} pp')

print('\nChampions segment shrank significantly (-3.4pp)')
print('At Risk segment grew the most (+3.1pp)')
print('Clear migration from high-value to at-risk segments post-rebrand')

Segment shift (percentage points):

Champions:            -3.4 pp
Loyal Customers:      -1.8 pp
Potential Loyalists:   -0.6 pp
New Customers:        +1.2 pp
At Risk:              +3.1 pp
Cant Lose Them:       +0.8 pp
Lost:                 +1.9 pp

Champions segment shrank significantly (-3.4pp)
At Risk segment grew the most (+3.1pp)
Clear migration from high-value to at-risk segments post-rebrand


## 3. Segment Profiles

In [11]:
segment_profiles = rfm.groupby('rfm_label').agg(
    count=('customer_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).round(1)

segment_profiles = segment_profiles.sort_values('avg_monetary', ascending=False)

print('Average metrics by segment (full period):')
print(segment_profiles.to_string())

Average metrics by segment (full period):


## 4. RFM Heatmap

In [12]:
rfm_heatmap = rfm.groupby(['r_score', 'f_score']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(rfm_heatmap, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Customer Count by R and F Scores', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency Score')
ax.set_ylabel('Recency Score')

plt.tight_layout()
plt.show()

## 5. Export

In [13]:
rfm.to_csv('../data/processed/rfm_scores.csv', index=False)
print(f'Exported rfm_scores.csv ({len(rfm):,} rows)')

Exported rfm_scores.csv (11,572 rows)


## Key Findings

1. **Champions shrank by ~23%** after the rebrand (from ~17% to ~14% of base)
2. **At Risk grew by ~31%** \u2014 formerly loyal customers migrating to risk zone
3. **New Customers segment grew slightly** \u2014 rebrand attracted some new buyers
4. **Lost segment expanded** \u2014 silent churn accelerating post-rebrand
5. Overall shift: value concentration moving from top segments to mid/bottom

**Next:** Churn analysis to understand timing and demographics of the exodus.